# **LLM As A Judge**

## **Initial issue**
I originally planned to use Mixtral on Hugging Face as my judge, but due to connectivity issues, I pivoted to an API-based architecture using GPT-4o-mini.

## **Judge Prompt**
To ensure the evaluation was more than just a surface-level check, I engineered a highly specific JUDGE_PROMPT that forced the AI to act as an expert Austrian tax law evaluator.

* I explicitly instructed the judge to reject any general tax knowledge and focus strictly on Austrian regulations like the EStG, UStG, and BAO. This was important to guarantee the model didn't accidentally reward 'German' tax logic, which is a common failure in LLMs.

* I added a strict rule to heavily penalize "hallucinated" or incorrect legal claims. If a model provided a wrong tax rate or a mistaken threshold, I capped its score at a 2, regardless of how well-written the rest of the answer was.

* Because professional tax advice should be precise, I told the judge to reward technical precision over word count and to penalize unnecessary introductory sentences.

I chose a 1-4 scale specifically to avoid **'central tendency bias.'** By removing the middle '3' in a 5-point scale, I forced the judge to make a definitive choice: is the answer legally sound (3 or 4) or is it legally deficient (1 or 2)? There should be no 'middle ground' when it comes to the correctness of a tax rate.

**Direct Scoring:** I used this for my project. The judge evaluates a single output's properties (like legal correctness) against a defined scale.

**Reference-Based Evaluation:** I used this approach for my assignment. The judge is provided with a "Gold Standard"

**Chain-of-Thought (CoT):** I commanded my judge to provide "Evaluation" reasoning before the score. Asking the LLM to think through the process step-by-step significantly improves evaluation quality and consistency.

## **BERTScore**

For the third assignment, I used BERTScore with baseline rescaling. I chose this because a regular BERTScore can be misleadingly high. Since most models know basic German grammar, they get high points just for 'sounding human' - I call these 'easy points.' By 'rescaling,' I am essentially subtracting those easy points so I can see the actual improvement each model provides. I chose BERTScore over traditional metrics like ROUGE because legal language is about meaning, not just exact word matching.

I also had to expand the provided dataset into dataset_clean_fixed.csv because I needed the 'Gold Standard' student answers to have a real baseline for comparison.

When I analyzed the results against my 'Gold Standard' reference, the specific metrics showed significant performance gap between the three models.

**BERTScores:**

**Model Architecture, Precision, Recall, F1 Score**
* **Inference (GPT-4o-mini)**, 0.0537, 0.3087, 0.1718
* **Fine-Tuned (Llama 3.1 8B)**, 0.3260, 0.2494, 0.2855
* **RAG (Gemma 1.1 2B)**, 0.2685, 0.1718, 0.2180

### **Explanation:**
* **The Baseline Inference Model:** Interestingly, the baseline had the highest recall (0.3087). However, this is misleading; it achieved this simply by being verbose. When you look at its extremely low precision of 0.0537, it becomes clear that while it covered a lot of 'ground truth', most of what it generated was legally irrelevant or non-specific.
* **The Fine-Tuned (SFT) Model:** With a precision of 0.3260, the fine-tuned model was the most effective at matching the specific expert dialect and professional phrasing found in the ground truth.
* **The RAG Model:** This models performance was constrained by the quality of the knowledge base (PDF about tax law). Its precision of 0.2685 is good, but its lower recall of 0.1718 reflects its tendency to refuse an answer rather than guess.


## **LLM Judge Evaluation**

The most interesting part of my project was the conflict between the scores.

**Model Architecture, Average LLM Judge Score (1-4), Key Performance Insight**

* **Inference (GPT-4o-mini)**, 2.34, Highest score due to high natural fluency and "Self-Enhancement" bias.
* **Fine-Tuned (Llama 3.1 8B)**, 1.5, Improved stylistic alignment (High BERTScore) but struggled with legal logic.
* **RAG (Gemma 1.1 2B)**, 1.33, Lowest score due to strict "Legal Humility" and safety-first refusals.

### **Explanation:**
* **The Fluency Trap:** My Inference model got the highest Judge score **(2.34)**. However, because GPT-4o-mini was used as both the generator and the judge, it likely favored its own architectural "voice" and formatting („Self-Enhancement Bias“).

* **Style vs. Logic:** My Fine-Tuned model (SFT) had the best BERTScore, but only a **1.5** from the judge. It learned how to 'talk' like a tax lawyer, but it didn't actually learn the law - it hallucinated paragraphs that don't exist.

* **Legal Humility:** The RAG model got the lowest score **(1.33)**. However, I argue this is my most successful model. It only scores low because it says 'Fail' when it can't find a source. In tax law, an honest refusal is much safer than a confident lie.

## **Final Conclusion**

My conclusion is that metrics like BERTScore aren't enough. We need expert-level audits to catch technical hallucinations. While fine-tuning makes a model sound better, RAG is the only architecture here that showed the legal humility necessary for real-world tax advisory.

In [ ]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import re  # Import regex module (NOTE: required because you use re.search later)
import time

client = OpenAI(api_key = "api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers
JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one.
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string

def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("inference_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION")  # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total = len(examples)):  # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question = row["prompt"],  # Insert question text
                reference = row["correct_answer"],  # Insert reference/legal correct answer
                answer = row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model = model,  # Specify model name
                messages =[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature = 0.2,  # Low temperature for consistent evaluation
                max_tokens = 400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK") # Print sample number
            print(response) # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1) # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:") # Print error header
            traceback.print_exc() # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_inf.csv", index = False)  # Save evaluation results to CSV file

if __name__ == "__main__": # Standard Python entry point check
    main() # Run main function when script is executed

"""
FINAL AVERAGE SCORE: 2.34
"""

File Error: [Errno 2] No such file or directory: 'dataset_clean_fixed.csv'


'\nFINAL AVERAGE SCORE: 2.34\n'

In [ ]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import time
import re  # Import regex module (NOTE: required because you use re.search later)

client = OpenAI(api_key = ""api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers
JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one.
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string


def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("fine_tuned_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn user if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION") # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total=len(examples)): # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question=row["prompt"],  # Insert question text
                reference=row["correct_answer"],  # Insert reference/legal correct answer
                answer=row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model=model,  # Specify model name
                messages=[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature=0.2,  # Low temperature for consistent evaluation
                max_tokens=400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK")  # Print sample number
            print(response)  # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1)  # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:")  # Print error header
            traceback.print_exc()  # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_ft.csv", index=False)  # Save evaluation results to CSV file

if __name__ == "__main__":  # Standard Python entry point check
    main()  # Run main function when script is executed

"""
FINAL AVERAGE SCORE: 1.5
"""

File Error: [Errno 2] No such file or directory: 'dataset_clean_fixed.csv'


'\nFINAL AVERAGE SCORE: 1.5\n'

In [ ]:
import pandas as pd
import time
from tqdm import tqdm  # Import tqdm for progress bar visualization in loops
import traceback  # Import traceback for detailed error printing
from openai import OpenAI  # Import OpenAI client to call the LLM API
import re  # Import regex module (NOTE: required because you use re.search later)

client = OpenAI(api_key = "api_key_here")  # Initialize OpenAI client with API key (replace "x" with real key)
model = "gpt-4o-mini"  # Define which OpenAI model to use for evaluation

# Define a large prompt template used to evaluate tax answers

JUDGE_PROMPT = """
You are an expert Austrian tax law evaluator.

Your task is to assess how accurately and correctly a system-generated answer responds to a tax law question under Austrian law.

You MUST evaluate strictly based on Austrian tax regulations (e.g. EStG, UStG, BAO etc.). Do not assume general tax knowledge is sufficient.

Inputs:
- TAX QUESTION
- REFERENCE ANSWER (legally correct answer)
- SYSTEM ANSWER (to be evaluated)

Evaluation criteria:

1. Legal correctness
- Must comply with Austrian tax law
- Check tax rates, thresholds, and legal rules

2. Jurisdiction accuracy
- Must be Austria-specific (not Germany or generic EU law)

3. Completeness
- Compare against reference answer coverage

4. Risk of misinformation
- Penalize hallucinated or incorrect legal claims heavily

Grading Scale (ONLY 1–4):

1 — Incorrect / Dangerous
- Legally wrong under Austrian tax law
- Misleading or hallucinated rules
- Wrong jurisdiction or tax rates
- Would lead to incorrect tax decisions

2 — Partially correct
- Some correct elements but important legal mistakes or omissions
- Mixed correct and incorrect Austrian law
- Potentially misleading

3 — Mostly correct
- Legally sound under Austrian tax law
- Minor omissions or lack of detail
- No major legal errors

4 — Fully correct / Excellent
- Completely correct under Austrian tax law
- Matches reference answer in legal substance
- Precise and applicable in practice

STRICT RULES:
- Be conservative: if unsure, lower the score
- Any legal inaccuracy cannot be 4
- Any wrong tax rate or threshold max score is 2
- Do not assume or guess tax rules
- Penalize confident but incorrect statements heavily
- Focus only on legal correctness
- A concise legal answer is better than a long one
- Penalize the model for unnecessary long or introductory sentences. The goal is technical precision, not word count.

OUTPUT FORMAT:
Evaluation: brief legal reasoning referencing Austrian law
Total rating: integer 1–4

Question:
{question}

Reference Answer:
{reference}

System Answer:
{answer}

Begin evaluation.

Evaluation:
"""  # End of prompt template string


def main():  # Define main execution function
    try:  # Start error handling block for file loading
        df_human = pd.read_csv("dataset_clean_fixed.csv", sep = ";")  # Load dataset with human/reference answers
        df_inference = pd.read_csv("rag_results.csv", sep = ";")  # Load model-generated answers

        df_human["id"] = df_human["id"].astype(str)  # Ensure ID column is string type for safe merging

        examples = pd.merge(df_human, df_inference, on="id").copy()  # Merge datasets on shared ID column

        if len(examples) == 0:  # Check if merge produced no rows
            print("Merge failed: No matching IDs")  # Warn user if merge failed
            return  # Exit function early

        print("Data loaded. Merged rows:", len(examples))  # Print number of merged samples

    except Exception as e:  # Catch any file reading or merge errors
        print("File Error:", e)  # Print error message
        return  # Exit program safely

    results = []  # Initialize list to store evaluation results

    print("STARTING LEGAL JUDGE EVALUATION")  # Notify start of evaluation loop

    for i, row in tqdm(examples.iterrows(), total=len(examples)):  # Loop through dataset with progress bar
        try:  # Start try block for API call safety

            prompt_content = JUDGE_PROMPT.format(  # Fill prompt template with row data
                question=row["prompt"],  # Insert question text
                reference=row["correct_answer"],  # Insert reference/legal correct answer
                answer=row["answer"]  # Insert system-generated answer
            )

            output = client.chat.completions.create(  # Call OpenAI chat completion API
                model=model,  # Specify model name
                messages=[  # Provide conversation context
                    {"role": "system", "content": "You are a strict Austrian tax law evaluator."},  # System role instruction
                    {"role": "user", "content": prompt_content}  # User prompt with evaluation task
                ],
                temperature=0.2,  # Low temperature for consistent evaluation
                max_tokens=400  # Limit output length
            )

            response = output.choices[0].message.content.strip()  # Extract and clean model response text

            print(f"\nSAMPLE {i + 1} FEEDBACK")  # Print sample number
            print(response)  # Print full evaluation output

            score_match = re.search(r"Total rating:\s*([1-4])", response)  # Extract numeric score using regex

            if score_match:  # If regex matched a score
                score = int(score_match.group(1))  # Convert extracted score to integer
            else:
                score = None  # If no score found, assign None

            results.append({"id": row["id"], "score": score})  # Store result for this row

            time.sleep(1)  # Pause to avoid rate limits

        except Exception:  # Catch API or runtime errors per row
            print("API Error:")  # Print error header
            traceback.print_exc()  # Print full stack trace for debugging
            results.append({"id": row["id"], "score": None})  # Store failed result

    df_final = pd.DataFrame(results)  # Convert results list into DataFrame
    avg_score = df_final["score"].mean()  # Compute average score across all evaluated samples

    print("\nFINAL AVERAGE SCORE:", round(avg_score, 2) if not pd.isna(avg_score) else "N/A")  # Print final score summary

    df_final.to_csv("judge_evaluation_rag.csv", index=False)  # Save evaluation results to CSV file

if __name__ == "__main__":  # Standard Python entry point check
    main()  # Run main function when script is executed

"""
FINAL AVERAGE SCORE: 1.33
"""

File Error: [Errno 2] No such file or directory: 'dataset_clean_fixed.csv'


'\nFINAL AVERAGE SCORE: 1.33\n'